# 03 – Fra piksler til prediksjoner

**Mål:** følge formene fra bilder til klassescores,
forstå vektede summer og bygge en lineær modell.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn


In [ ]:
from pytorch_101.utils import download_mnist

train_data, test_data = download_mnist()
print(f"Treningsbilder: {len(train_data)}, testbilder: {len(test_data)}")


## Eksempel: en vektet sum

En lineær modell multipliserer hver input med en vekt, summerer og
legger til et konstantledd (*bias*). Vektene bestemmer påvirkningen fra hver verdi.


Den vektede summen skrives som `z = x @ W + b`, der `@` er matrisemultiplikasjon, `x` input, `W` vekter, og `b` bias.


In [ ]:
# To målinger gir én score.
x = torch.tensor([[2.0, 3.0]])
weights = torch.tensor([[0.5], [-1.0]])
bias = torch.tensor([0.2])
score = x @ weights + bias
print("Score:", score, "Form:", score.shape)
print("Samme beregning for hånd:", 2 * 0.5 + 3 * (-1) + 0.2)


## Oppgave 1: Endre én vekt og forutsi virkningen

La
`inputs = [[1., 2., 3.], [4., 5., 6.]]`,
`weights = [[1., 0.], [0., 1.], [1., -1.]]`, `bias = [0.5, -0.5]`.
Definer tensorene og beregn `scores`. Øk så vekten i rad 0, kolonne 0 med 1
og beregn `changed_scores`. Hvilke utganger endres, og med hvor mye?
Skriv forventningen i svarcellen før kjøring.

<details><summary>Hint</summary>

Kopier vektene med `clone()`. Vektens rad velger inngang, kolonnen velger utgang.

</details>


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert torch.allclose(scores, torch.tensor([[4.5, -1.5], [10.5, -1.5]]))

assert torch.allclose(changed_scores - scores, torch.tensor([[1.0, 0.0], [4.0, 0.0]]))


### Refleksjon

Hva representerer radene og kolonnene i resultatet? Hvorfor må de to indre
dimensjonene stemme ved matrisemultiplikasjon? Hvordan ville `*` oppført seg her?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.

Hvilken klassekolonne endret seg? Knytt størrelsen på endringen til inngangsverdiene.


**Mitt svar:**

Skriv din forklaring her.


## Eksempel: flat ut bilder, behold bildeskillet

En lineær modell trenger én rekke tall per bilde. `flatten(start_dim=1)`
beholder dimensjon 0 og slår sammen resten. Piksler fjernes ikke, men den
todimensjonale plasseringen er ikke lenger eksplisitt i formen.


In [ ]:
small_images = torch.arange(24, dtype=torch.float32).reshape(2, 1, 3, 4)
flattened = small_images.flatten(start_dim=1)
print(small_images.shape, "→", flattened.shape)


## Nå trenger vi grupper av bilder

Til nå har vi hentet enkeltbilder med `train_data[index]`. Modellen
kan regne på flere bilder samtidig, og under trening vil vi oppdatere
den etter hver lille gruppe. En slik gruppe kalles en **batch**.

`DataLoader` samler bilder og klasser fra datasettet. 

`batch_size=64` gir opptil* 64 bilder, og `shuffle=True`
stokker rekkefølgen for hver gjennomgang.


Datasettet inneholder eksemplene; lasteren bestemmer hvordan de grupperes.
`next(iter(...))` henter én gruppe som vi kan undersøke.

*Den siste batchen kan inneholde færre dersom len(train_data) $\bmod$ batch_size $\neq$ 0


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
images, labels = next(iter(train_loader))
print("Bilder:", images.shape, images.dtype)
print("Etiketter:", labels.shape, labels.dtype)


### Refleksjon

Forklar hvert tall i bildegruppens form. Hvorfor har klassene bare
én dimensjon? Hva ville endret seg om `batch_size` var 16?

Skriv svaret ditt i Markdown-cellen under.


**Mitt svar:**

Skriv din forklaring her.


## Oppgave 2: Lag ti scores for MNIST

Hent `images, labels` fra `train_loader`. Lag `flat_images` med én rad
per bilde. Lag tilfeldige `mnist_weights` med formen `[784, 10]`, gang dem med
0.01, og lag `mnist_bias` som ti nuller. Beregn `mnist_scores`.
Bruk `argmax(dim=1)` til å lage `predictions`. Skriv ut alle formene.

<details>
<summary>Hint</summary>

`torch.randn` lager tilfeldige tall. `argmax(dim=1)` finner kolonnen med høyest score i hver rad.

</details>


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert flat_images.shape == (images.shape[0], 784)
assert mnist_scores.shape == (images.shape[0], 10)
assert predictions.shape == labels.shape


## Eksempel: la `nn.Linear` håndtere parametrene

`nn.Linear(3, 2)` lærer tre vekter og ett konstantledd per utgang.
Den lagrer vektene som `[utganger, inputs]`, motsatt av vår håndlagde
matrise. Laget utfører den nødvendige transponeringen selv.
`nn.Sequential` kjører flere lag etter hverandre.

Med vår matrisenotasjon er dette `X @ layer.weight.T + layer.bias`.
Antall parametere er `inputs * utganger + utganger`.


In [ ]:
small_layer = nn.Linear(3, 2)
print("Resultat:", small_layer(torch.ones(4, 3)).shape)
print("Vekter:", small_layer.weight.shape, "Konstantledd:", small_layer.bias.shape)


## Oppgave 3: Bygg en lineær bildemodell

Lag `model = nn.Sequential(...)` med `nn.Flatten()` og et passende `nn.Linear`.
Modellen skal motta bildene direkte og gi ti scores per bilde.
Beregn `model_scores`, og tell parametrene med
`sum(parameter.numel() for parameter in model.parameters())`.

<details>
<summary>Hint</summary>

Hvert bilde har `28 * 28 = 784` piksler. Det er ti mulige sifre.

</details>


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert model_scores.shape == (len(images), 10)
assert sum(p.numel() for p in model.parameters()) == 7850


### Refleksjon

Skriv formene gjennom modellen for en gruppe på 32 MNIST-bilder.
Hvorfor har modellen 7 850 parametere? Hvorfor forventer vi ikke gode
prediksjoner før trening? Er de ti scorene sannsynligheter?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Skriv din forklaring her.


## Referanser


- [PyTorch – Build the Neural Network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html).
- [3Blue1Brown – But what is a Neural Network?](https://www.3blue1brown.com/lessons/neural-networks/) - Visuell introduksjon til nevrale nettverk.
